In [ ]:


## Preparing Workspace -----------------------------------------------------------------------------------------------------------


## Importing packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
from tqdm.auto import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
from IPython.display import display



# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'



## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())

## Setting API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
file_api = path_config / 'api_key.txt'
with open(file_api, 'r') as file:
    api_key = file.read()




In [ ]:

import yaml

file_in = path_config / 'census_configuration_file2.xlsx'
df_acs = pd.read_excel(file_in, sheet_name='ACS')
# df_acs = df_acs[df_acs['Indicator Name'] == 'Commute_1']



# df_acs = df_acs.set_index(['Indicator Name'])
# dict_acs = df_acs.T.to_dict()
# dict_acs = df_acs.to_dict(orient='records')
# dict_acs




# file_out = path_config / 'test.yaml' 
# with open(file_out, 'w') as f:
#     yaml.dump(dict_acs, f, default_flow_style=False)


In [ ]:


print(); print()
print('Census Bureau import parameters:')
print()


path_yaml = path_config0 / 'config_indicators.yaml'

try:
    with open(path_yaml, 'r') as yaml_file:
        dict_config = yaml.load(yaml_file, Loader=yaml.SafeLoader)
except FileNotFoundError:
    print(f"Error: The file at {path_yaml} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

print('---------------------------------------------------------------------------------------------------------------------------------------')
print()
print('Projects available: ')
display(dict_config['Project']); print()
print('Which project are you pulling data for?'); print()
project = input()
print()
print('---------------------------------------------------------------------------------------------------------------------------------------')

print()
print('Indicators available:'); print()
display(list(dict_config['Indicators'][project].keys())); print()
print('Which indicator do you need to rerun?'); print()
indicator_name = input()
print()
print('---------------------------------------------------------------------------------------------------------------------------------------')

print()
display(dict_config['Indicators'][project][indicator_name])
export_loc  =     dict_config['Indicators'][project][indicator_name]['sp_location'        ]
folder      =     dict_config['Indicators'][project][indicator_name]['folder'             ]
sample_type =     dict_config['Indicators'][project][indicator_name]['sample'             ]
MOE_thresh  = int(dict_config['Indicators'][project][indicator_name]['MOE_threshold'      ])
num_vars    = int(dict_config['Indicators'][project][indicator_name]['number_of_variables'])

print()
print('Estimates available:')
display(list(dict_config['Samples'][sample_type].keys())); print()
print('Which estimate do you want to pull data from?'); print()
estimate = input()
print()
print('---------------------------------------------------------------------------------------------------------------------------------------')


print()
print('Geographies available:'); print()
display(dict_config['Samples'][sample_type][estimate]['geographies_available']); print()
print('Which geography do you want to pull data for?'); print()
geography = input()
print()
print('---------------------------------------------------------------------------------------------------------------------------------------')

print()
print('Years available:')
print(dict_config['Samples'][sample_type][estimate]['years_available']); print()
print('Do you want to pull data for all years available?  Select Yes/No: '); print()
all_years = input()
if all_years == 'Yes':
    years_to_import = dict_config['Samples'][sample_type][estimate]['years_available']
if all_years == 'No':
    print()
    print('Please type which years you want to pull data from, separated by commas:'); print()
    years_to_import = input()
    if ',' in years_to_import:
        years_to_import = years_to_import.split(', ')
    else:
        years_to_import = list(years_to_import)
    years_to_import = [int(year) for year in years_to_import]
print()
import_tab = dict_config['Import Geographies'][geography]

print('---------------------------------------------------------------------------------------------------------------------------------------')

